<a href="https://colab.research.google.com/github/AdityaJare/ML-PRojects/blob/main/intent_detection_distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/test.csv
/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/train.csv
/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/valid.csv


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, Trainer, TrainingArguments, DistilBertForSequenceClassification
import re
import nltk

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/train.csv")

test = pd.read_csv("/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/test.csv")

valid = pd.read_csv("/kaggle/input/datasets/himanshunayal/intent-recognition-dataset/valid.csv")

In [ ]:
print(f"train : {train.shape}")
print(f"test : {test.shape}")
print(f"validation : {valid.shape}")

train : (13084, 2)
test : (700, 2)
validation : (700, 2)


In [ ]:
train["intent"].value_counts()

intent
PlayMusic               1914
GetWeather              1896
BookRestaurant          1881
RateBook                1876
SearchScreeningEvent    1852
SearchCreativeWork      1847
AddToPlaylist           1818
Name: count, dtype: int64

In [ ]:
train.head()

,text,intent
0,listen to westbam alumb allergic on google music,PlayMusic
1,add step to me to the 50 clásicos playlist,AddToPlaylist
2,i give this current textbook a rating value of...,RateBook
3,play the song little robin redbreast,PlayMusic
4,please add iris dement to my playlist this is ...,AddToPlaylist


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13084 entries, 0 to 13083
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    13084 non-null  object
 1   intent  13084 non-null  object
dtypes: object(2)
memory usage: 204.6+ KB


In [ ]:
train.isnull().sum()

text      0
intent    0
dtype: int64

In [ ]:
train["intent"].unique()

array(['PlayMusic', 'AddToPlaylist', 'RateBook', 'SearchScreeningEvent',
       'BookRestaurant', 'GetWeather', 'SearchCreativeWork'], dtype=object)

In [ ]:
le = LabelEncoder()

train["label"] = le.fit_transform(train["intent"])

test["label"] = le.transform(test["intent"])

valid["label"] = le.transform(valid["intent"])

In [ ]:
train

,text,intent,label
0,listen to westbam alumb allergic on google music,PlayMusic,3
1,add step to me to the 50 clásicos playlist,AddToPlaylist,0
2,i give this current textbook a rating value of...,RateBook,4
3,play the song little robin redbreast,PlayMusic,3
4,please add iris dement to my playlist this is ...,AddToPlaylist,0
...,...,...,...
13079,i want to eat choucroute at a brasserie for 8,BookRestaurant,1
13080,play funky heavy bluesy,PlayMusic,3
13081,rate the current album 2 points out of 6,RateBook,4
13082,go to the photograph the inflated tear,SearchCreativeWork,5


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
sample = train["text"][0]
tokens = tokenizer(sample)
print(sample)
tokens


listen to westbam alumb allergic on google music


{'input_ids': [101, 4952, 2000, 2225, 3676, 2213, 2632, 25438, 27395, 2006, 8224, 2189, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

Tokens: ['[CLS]', 'listen', 'to', 'west', '##ba', '##m', 'al', '##umb', 'allergic', 'on', 'google', 'music', '[SEP]']


In [ ]:
train_encoding = tokenizer(
    train["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

test_encoding = tokenizer(
    test["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

valid_encoding = tokenizer(
    valid["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

In [ ]:
train_encoding[0]

Encoding(num_tokens=38, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [ ]:
train_encoding["input_ids"][0]

tensor([  101,  4952,  2000,  2225,  3676,  2213,  2632, 25438, 27395,  2006,
         8224,  2189,   102,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [ ]:
train_dataset = Dataset.from_dict({"input_ids" : train_encoding["input_ids"], "attention_mask": train_encoding["attention_mask"], "labels":train["label"].tolist()})

test_dataset = Dataset.from_dict({"input_ids" : test_encoding["input_ids"], "attention_mask": test_encoding["attention_mask"], "labels":test["label"].tolist()})

valid_dataset = Dataset.from_dict({"input_ids" : valid_encoding["input_ids"], "attention_mask": valid_encoding["attention_mask"], "labels":valid["label"].tolist()})

In [ ]:
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 13084
})

In [ ]:
print(len(train_dataset))
print(len(valid_dataset))
print(len(test_dataset))

13084
700
700


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = len(le.classes_)
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model.classifier.out_features)

7


In [ ]:
training_arguments = TrainingArguments(
    output_dir="./intent_detection_results",
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end = True,
    logging_steps=10,
    fp16 = True
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_arguments,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.155428,0.151643
2,0.092505,0.130942
3,0.009228,0.128586


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1227, training_loss=0.23759121061306518, metrics={'train_runtime': 141.7223, 'train_samples_per_second': 276.964, 'train_steps_per_second': 8.658, 'total_flos': 385942989989808.0, 'train_loss': 0.23759121061306518, 'epoch': 3.0})

In [ ]:
test_result = trainer.evaluate(test_dataset)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch
0.009228,0.167944,3


In [ ]:
print(test_result)

{'eval_loss': 0.16794362664222717}


In [ ]:
text  ="I want to listen golden blown for 1 hour contineously"

In [ ]:
import torch

inputs = tokenizer(
    text,
    padding= True,
    truncation=True,
    return_tensors = "pt"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k : v.to(device) for k, v in inputs.items()}
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-1.5382, -1.9733, -1.6784,  5.3906, -2.0782, -1.1411, -0.1904]],
       device='cuda:0'), hidden_states=None, attentions=None)

In [ ]:
logits = outputs.logits

In [ ]:
predicted_class_id = torch.argmax(logits, dim= -1).item()

In [ ]:
 predicted_intent = le.inverse_transform([predicted_class_id])[0]

In [ ]:
print(f"Predicted Intent: {predicted_intent}")

Predicted Intent: PlayMusic
